# Pokémon + MTG: LoRA (UNet) + Textual Inversion (TI) — End‑to‑End Tutorial

## Motivation

Modern image models can turn a short prompt into rich art. Most use a **text encoder** (e.g., CLIP — *Contrastive Language–Image Pretraining*) to map prompts to vectors that a generator (e.g., Stable Diffusion’s UNet) understands. Example prompts like “an astronaut riding a horse” work well because the model already knows those concepts:</br></br>
<img src="assets/astronaut_rides_horse.png" alt="Astronaut riding a horse — base model example" width="256"/></br></br>

When you need **a specific IP/style** or **creatures that don’t exist**, base models struggle. Ask a generic model for “Bugs Bunny playing baseball” and you’ll likely get a bunny‑ish cartoon — not the exact character — and the style may drift.
</br></br>
<img src="assets/bugs_playing_baseball.png" alt="Bugs Bunny playing baseball — base model example" width="256"/></br>
Thus, without model adaptation, the IP or identity is not reproduced; instead you get a generic look-alike.</br>

For this tutorial, you’ll build **Magic: The Gathering–style card art** of **Pokémon characters**. We’ll combine two lightweight adaptations so prompts become precise and stylized:

* **Style LoRA (UNet):** teaches Stable Diffusion the **Pokémon/MTG visual manifold** (poses, lighting, linework, materials). Mentioning “pokemon” pushes the rendering into this look.
* **Per‑character Textual Inversion (TI):** gives each Pokémon a **unique trigger token** (e.g., `<pk_absol>`) that binds its **identity/shape** in the text encoder.

You can then write prompts like:

> `"<pk_snorlax> and <pk_weedle> pokemon, tug of war, card art, dramatic lighting"`




## What you’ll assemble

1. **Card‑art style** for Pokémon via a UNet **LoRA** adapter (global look/feel).
2. **Identity triggers** via **textual inversion** for each Pokémon (per‑character tokens).
3. **Scene prompts** that specify MTG‑like composition, background, and mood.
4. **Blend** (1)–(3) at inference to render specific Pokémon in a cohesive MTG style.

> Note: If you also want a formal MTG **creature type** (e.g., “Beast”, “Spirit”), you can simply include it in the prompt/preset; no classifier is required unless you’re auto‑generating card text.

### Architecture at a glance

```
Prompt ("<pk_name> pokemon, …")
        │
        ▼
Text Encoder (CLIP)
  ├─ Base weights (frozen)
  ├─ + TI token embeddings  ← per‑Pokémon identity vectors
  └─ (+ optional TE LoRA)   ← only if you add small TE adapters later
        │  (embeddings)
        ▼
UNet (diffusion denoiser)
  ├─ Base SD1.5 weights (frozen)
  └─ + Style LoRA (UNet)    ← global "pokemon / card‑art" look
        │  (conditioning)
        ▼
Latent → VAE Decode → Image (MTG‑style Pokémon art)
```
<ul>
<li>The user writes a prompt that includes the Pokémon’s trigger token (e.g., <pk_absol>), the class word “pokemon”, and desired style (e.g., “card art, dramatic lighting”).</li>
<li>The tokenizer splits the text into tokens; the CLIP text encoder (frozen base plus your TI token embeddings) produces the text embeddings that condition the generator.</li>
<li>The UNet denoiser starts from random latent noise and iteratively denoises it toward an image latent, using cross-attention to the text embeddings. It uses the base Stable Diffusion weights augmented by your style LoRA (UNet) adapter.</li>
<li>The final latent is passed through the VAE decoder to yield the RGB image.</li></ul>

### Data Layout

* Source images: `datasets/pokemon-images/<name>/*.jpg|png` (one folder per Pokémon).
* Destination images: `results/pokemon-images/`
* Embedded weights: `results/pokemon-images/<name>.pt` (one weights file per Pokémon).


---


## Applications
Our tool has application in both Text-to-Image generation and Image-to-Image generation
### Text‑to‑Image (t2i)
Generate an image based on a text prompt description.

```
random latent x_T  ──►  UNet denoiser (T→0 steps, conditioned on CLIP text)  ──►  clean latent x_0  ──►  VAE decoder  ──►  RGB image
                   (latent space: 4×H/8×W/8)
```

* Starts from **noise in latent space** (no VAE encoder).
* **UNet** iteratively denoises the latent using **text embeddings** from CLIP.
* **VAE decoder** converts the final latent to pixels.

### Image‑to‑Image (i2i)
Adopt a style and/or background to an existing image.
```
input image  ──►  VAE encoder  ──►  latent x_0  ──►  add noise (strength) ──►  UNet denoiser (k steps) ──►  latent x_k  ──►  VAE decoder ──► image
```

* Starts from an **encoded latent** of your image (via **VAE encoder**).
* Noise amount controls how much the result keeps the original structure.
* The rest matches t2i: **UNet denoises**; **decoder** renders pixels.

> In both flows, your **Style LoRA (UNet)** affects denoising style; **TI tokens / TE LoRA** affect text conditioning.


## Procedure
### Data Collection
#### Create Pokemon text descriptors (POKEDESC)
To train Textual Inversion (TI) to images and terms foreign to the existing CLIP model, we need consistent text description of the creature. This provides these benefits:
<ul><li>Consistent conditioning: CLIP learns from text–image pairs. Feeding stable fields like body and colors set anchors the embedding to the right visual attributes.</li>
<li>Disentangles style: Keep painting style in STYLE_PRESETS (or separate class prompts) so the token learns the subject, not the renderer.</li>
<li>Better generalization: Using the same concise anatomy/palette phrases across varied poses/backgrounds teaches the token what’s essential vs incidental.</li></ul>

In [1]:
from typing import TypedDict, Dict, Iterable, Optional, List, Tuple
from pathlib import Path
import re
import requests, html
import json
import traceback

CREATE_POKEDESC = False
CREATE_DATASET = False

OUT_JSON = Path("./results/pokemon-images/pokedesc_autofill.json")

# Define set of Pokemon to learn
POKEMON = ['Jirachi', 'Staraptor', 'Bewear', 'Krookodile', 'Bayleef', 'Kingler', 'Mismagius', 'Magmar','Magikarp',
 'Gliscor', 'Zacian', 'Houndoom', 'Ampharos', 'Fuecoco', 'Hydreigon', 'Lopunny', 'Golem','Weedle', 'Scorbunny',
 'Togepi', 'Cinderace', 'Electivire', 'Weavile', 'Luxray', 'Froslass','Deoxys', 'Absol','Gholdengo',
 'Bisharp', 'Marowak','Tapu Koko', 'Amaura', 'Annihilape', 'Grimmsnarl', 'Quagsire', 'Zeraora', 'Crobat', 'Darkrai', 'Totodile', 'Dragapult',
 'Psyduck', 'Skarmory', 'Tinkaton', 'Ditto', 'Ho-Oh', 'Jigglypuff', 'Kyurem', 'Torterra', 'Venusaur',
 'Haxorus','Lycanroc', 'Chandelure', 'Nidoking', 'Slowpoke', 'Suicune', 'Yamper', 'Vulpix',
 'Incineroar',"Sirfetch'd", 'Articuno', 'Salamence', 'Meowth', 'Wobbuffet','Aegislash', 'Alakazam',
 'Piplup', 'Lapras', 'Zoroark', 'Goodra', 'Flygon', 'Lugia', 'Mudkip', 'Gyarados',
 'Scizor', 'Typhlosion', 'Heracross', 'Metagross', 'Infernape', 'Excadrill', 'Dragonite', 'Arcanine',
 'Mew', 'Sceptile', 'Arceus', 'Tyranitar', 'Rowlet', 'Garchomp', 'Snorlax', 'Blaziken',
 'Mimikyu', 'Squirtle', 'Gardevoir', 'Bulbasaur', 'Rayquaza', 'Gengar', 'Mewtwo', 'Eevee',
 'Greninja', 'Lucario', 'Charizard', 'Pikachu', 'Blastoise']

# The POKEDESC is a JSON doc with a Pokemon name mapping to a PokeDetails class
class PokeDetails(TypedDict):
    form: str    # one-word description of appearance
    body: str    # general shape
    colors: int  # main colors
    mood: str    # mood appearance "joyful", "menacing", etc.
    extras: str  # other useful features

In [2]:
# Simple word classification to parse phrases to key sentences and features
    # Adjectives which typically go with body parts
ADJ = {"small","large","huge","long","short","pointed","rounded","round","slender","thick", "tiny"}
    # Body parts
ANATOMY_WORDS = {
    "head","face","belly","back","chest","neck","muzzle","snout","jaw","ear","eye","antenna", "plate","antennae",
    "horn","crest","plume","mane","frill","fin","fins","wing","feather","tail","claw", "tip", "underside",
    "spike","spines","quill","quills","shell","carapace","scale","fur","coat","pelt","whisker","teeth","fang","tusk"
}

APPEARANCE_WORDS = set("""
color coloured colored coloration covered hue pattern stripe spotted speckled markings banded horn horns antler crest plume mane frill fin fins
wing wings feather beak muzzle snout jaw tusk tooth teeth fang fangs ear ears eye eyes pupil iris rodent
scale scales scaly carapace shell armor armour plating spike spikes spines quill quills fur furry coat pelt hair hairy whisker
tail tails claw claws talon talons paw paws hand hands foot feet limb limbs arm arms leg legs digit
body torso back belly chest neck head skull dome shell carapace carapaced carapacial
height tall short long slender thin lithe bulky stout muscular stocky massive huge small tiny
biped bipedal quadruped quadrupedal serpentine segmented insectoid avian feline canine ursine draconic humanoid
metal metallic steel rocky stone gaseous ghostly translucent transparent crystalline crystal icy molten 
mammal mammalian dark
""".split())

# Behavior/ability/habitat etc. -- words that do not describe appearance
BEHAVIOR_WORDS = set("""
aggressive calm timid friendly hostile migratory migrates migrates seasonally hunts hunts at night preys prey predator territorial
lives dwells inhabits habitat nest nests burrow burrows lair cave forest ocean river lake mountain city village
feeds feeding diet eats drinking sleep sleeps nocturnal diurnal crepuscular communicates calls song dances mating courtship breeds breeding reproduces
battle battles fights attacks attack defends defense defensive offensive uses can able capable emits shoots breathes fires launches generates
ability move move-set moveset movepool pressure intimidate blaze torrent overgrow levitate synchronize
trainer trainers evolves evolution evolves into mega gigamax gigantamax regional forme form forms
""".split())

BODYPLAN_WORDS = set("""
biped bipedal quadruped quadrupedal serpentine avian insectoid mammal mammalian feline canine ursine draconic humanoid turtle
""".split())

COLOR_WORDS = {'amber', 'beige', 'black', 'blue', 'blue-green', 'brown', 'cream' 'cream-colored', 'crimson', 'cyan', 'dark blue', 
               'dark green', 'gold', 'gray', 'green', 'grey', 'icy-blue', 'ivory', 'light blue', 'magenta', 'maroon', 'molten-red', 
               'navy', 'orange', 'pink', 'purple', 'red', 'tan', 'teal', 'turquoise', 'violet', 'white', 'yellow'}

ICONIC_EXTRAS = {
    # things you'd like in "extras" (short fragments)
    "crest","plume","mane","frill","fins","wings","feathers","horns","spikes","spines","quills",
    "shell","carapace","scales","pattern","stripes","spots","banded"
}

PATTERN_HEADS = {"stripes","spots","back","markings","bands"} 

# Quantity and direction words
QUANT_DIR = {"one","two","three","four","horizontal","vertical","diagonal","bold","thin","thick"}

SHAPE_WORDS = {'bulky', 'huge', 'large', 'long', 'massive', 'pointed', 'round',
     'rounded', 'short', 'slender', 'small', 'stocky', 'thick', 'tiny'}

In [3]:

# Defined words converted to be placed in a regex statement. Some words have dashes which need to be escaped

def nonCapture(regex: str):
    ''' convert a regex string into a non-capturing group. '''
    return "(?:" + regex + ")"
ADJ_REGEX = "|".join([s.replace("-","\-") for s in ADJ]) 
ADJ_COLOR_REGEX = "|".join([s.replace("-","\-") for s in ADJ|COLOR_WORDS]) 
    # for body parts also allow for 's' to support plural. (doesn't work for antennae)
ANATOMY_REGEX = 's?|'.join([s.replace("-","\-") for s in ANATOMY_WORDS]) + "s?"
    # for body phrase allow commas between adjective and anatomy
BODY_PHRASE_REGEX = re.compile(
    rf"\b((?:(?:{nonCapture(ADJ_REGEX)})(?:\s*,\s*|\s+))*)({nonCapture(ANATOMY_REGEX)})\b",
    re.I,)
BODYPLAN_REGEX = "|".join([s.replace("-","\-") for s in BODYPLAN_WORDS])
COLOR_REGEX = '|'.join([s.replace("-","\-") for s in COLOR_WORDS])
COLOR_PHRASE_REGEX = re.compile(rf"\b(({nonCapture(COLOR_REGEX)})\s+({ANATOMY_REGEX}))\b", re.I)
QUANT_DIR_REGEX = '|'.join([s.replace("-","\-") for s in QUANT_DIR])
SHAPE_REGEX = '|'.join([s.replace("-","\-") for s in SHAPE_WORDS])


# regex patterns to clean up sentences
_PAREN_RE = re.compile(r"\s*\([^)]*\)")
_SPACE_RE = re.compile(r"\s+")
_CIT_RE   = re.compile(r"\s*\[[^\]]*\]")
_END_PUNC_RE = re.compile(r"[.!?]\s*$")


In [4]:
STYLE_PRESETS: Dict[str, Dict[str, str]] = {
    "3d": {
        "positive": "3d render, studio lighting, high detail, global illumination, subsurface scattering",
        "negative": "low quality, blurry, deformed, extra limbs, worst quality, jpeg artifacts",
    },
    "watercolor": {
        "positive": "watercolor painting, textured paper, soft brush strokes, high detail",
        "negative": "cartoonish outline, heavy posterization, low-res, artifacts",
    },
    "card_art": {
        "positive": "illustration, fantasy trading card art, dramatic lighting, intricate details, professional artstation style",
        "negative": "amateur, messy, low detail, anatomy errors, watermark, signature",
    },
}

In [5]:
# Collect the biology section of bulbapedia for a particular pokemon to understand the appearance
def bulbapedia_biology(name: str, session: Optional[requests.Session] = None) -> Optional[str]:
    """
    Return Bulbapedia's 'Biology' section (appearance description) for e.g. 'Gyarados'. Also gives general form.
    Why: human-written look/shape details vs. behavior-only PokéAPI flavor_text.
    """
    s = session or requests.Session()
    title = f"{name} (Pokémon)"
    api = "https://bulbapedia.bulbagarden.net/w/api.php"

    # 1) find section index for "Biology"
    r1 = s.get(api, params={"action": "parse", "page": title, "prop": "sections", "format": "json"}, timeout=20)
    if r1.status_code != 200 or "parse" not in r1.json():
        return None
    sections = r1.json()["parse"].get("sections", [])
    bio_idx = None
    for sec in sections:
        if sec.get("line","").lower() == "biology":
            bio_idx = sec.get("index")
            break
    if not bio_idx:
        return None

    # 2) fetch HTML for the Biology section, then strip to plain text
    r2 = s.get(api, params={"action": "parse", "page": title, "prop": "text", "section": bio_idx, "format": "json"}, timeout=20)
    if r2.status_code != 200 or "parse" not in r2.json():
        return None
    html_text = r2.json()["parse"]["text"]["*"]

    # crude HTML → text cleanup (avoid extra deps)
    text = re.sub(r"<(script|style)[^>]*>.*?</\1>", "", html_text, flags=re.S|re.I)
    text = re.sub(r"<sup[^>]*>.*?</sup>", "", text)   # drop citation markers
    text = re.sub(r"<[^>]+>", " ", text)              # strip tags
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s*[-]\s*","-", text)                 # remove spaces around dashes
    text = html.unescape(text)
    startChar = text.find("source ]")+8
    text = text[startChar:].strip()
    return text or None


# Convert the long text description into 2 relevant sentences. Prioritize phrases which have colors, describe appearance, etc.

def _sentence_split(text: str) -> List[str]:
    # Conservative sentence split; keeps abbreviations mostly intact
    # Normalize weird spacing first
    txt = _SPACE_RE.sub(" ", text).strip()
    # Split on . ! ? followed by space+capital or end
    parts = re.split(r"(?<=[.!?;])\s+(?=[a-zA-Z(])", txt)
    # Clean citations/parentheticals in each sentence
    out = []
    for s in parts:
        s = _CIT_RE.sub("", s)
        s = _PAREN_RE.sub("", s)
        s = _SPACE_RE.sub(" ", s).strip()
        if s:
            out.append(s)
    return out

def _score_sentence(s: str) -> tuple[float, dict]:
    words = [w.lower() for w in re.findall(r"[A-Za-z][A-Za-z\-']+", s)]
    wset = set(words)
    app = len(wset & APPEARANCE_WORDS)
    beh = len(wset & BEHAVIOR_WORDS)

    length = len(words)
    if length == 0:
        return (-999.0, {"app":0,"beh":0,"len":0})

    # gentler length penalty (center at 22 words)
    length_center = 22
    length_pen = abs(length - length_center) / 18.0  # was /10; softer

    score = app - 1.5*beh - length_pen

    has_color = bool(wset & COLOR_WORDS)
    has_bodyplan = bool(wset & BODYPLAN_WORDS)
    if has_color and has_bodyplan:
        score += 1.2  # small but decisive bump
    elif has_color or has_bodyplan:
        score += 0.5

    return (score, {"app":app, "beh":beh, "len":length})

def _dedupe_selection(cands: List[str]) -> List[str]:
    # Keep diverse info: penalize near-duplicate content
    chosen: List[str] = []
    for s in cands:
        sw = set(w.lower() for w in re.findall(r"[A-Za-z][A-Za-z\-']+", s))
        if any(len(sw & set(w.lower() for w in re.findall(r"[A-Za-z][A-Za-z\-']+", t))) / max(1, len(sw)) > 0.6 for t in chosen):
            continue
        chosen.append(s)
        if len(chosen) == 2:
            break
    return chosen

def summarize_appearance_to_two_sentences(biology_text: str) -> str:
    """
    Reduce Bulbapedia Biology text to 2 sentences emphasizing physical appearance
    (anatomy, color, size/shape, materials), excluding behavior/habitat/moves.
    """
    sentences = _sentence_split(biology_text)
    if not sentences:
        return ""

    # Score & sort
    scored = []
    for s in sentences:
        score, meta = _score_sentence(s)
        # require at least some appearance signal if possible
        if meta["app"] == 0 and meta["beh"] > 0:
            score -= 1.0
        scored.append((score, s))
    scored.sort(key=lambda x: x[0], reverse=True)

    # Gather candidates with positive tendency; fallback to top
    candidates = [s for sc,s in scored if sc > 0] or [s for _,s in scored[:5]]
    picked = _dedupe_selection(candidates) or candidates[:2]

    # Clean up: ensure terminal punctuation and concise spacing
    cleaned = []
    for s in picked:
        s = s.strip()
        if not _END_PUNC_RE.search(s):
            s += "."
        cleaned.append(s)
    # If only one sentence survived, append next best distinct one
    if len(cleaned) == 1 and len(scored) > 1:
        fallback = next((s for _,s in scored if s != cleaned[0]), None)
        if fallback:
            if not _END_PUNC_RE.search(fallback): fallback += "."
            cleaned.append(fallback)

    return cleaned[:2]

# --- return 2-sentence description at once ---
def appearance_summary_from_bulbapedia(name: str) -> Optional[str]:
    bio = bulbapedia_biology(name)
    if not bio:
        return None
    pat = r'(\w+)\s+(?=Pok[eé]mon)'
    form = re.search(pat, bio).group().strip()
    return form, summarize_appearance_to_two_sentences(bio)


In [6]:
# Some test cases for Pokemon text description generation. See the output of bulbapedia_biology and the 2 sentence selection in the appearance_summary
if 1:
    pokemon = "Charizard"
    pokemonPhrase = bulbapedia_biology(pokemon)
    print(pokemonPhrase)
    form, summary = appearance_summary_from_bulbapedia(pokemon)
    print(form)
    print(summary)

Charizard's tail flame burning blue after Professor Oak accidentally steps on its tail Charizard is a draconic , bipedal Pokémon . It is primarily orange with a cream underside from the chest to the tip of its tail. It has a long neck, small blue eyes, slightly raised nostrils, and two horn-like structures protruding from the back of its rectangular head. Two fangs are visible in its upper jaw when its mouth is closed. Two large wings with blue-green undersides sprout from its back, and a horn-like appendage juts out from the top of the third joint of each wing. A single wing finger is visible through the center of each wing membrane. Charizard's arms are short and skinny compared to its robust belly, and each limb has three white claws. In Pokémon Red and Green , Charizard is seemingly shown holding a fireball, implying that it's capable of manipulating its flames using its hands; this is seen in media where it uses Fire Punch . It has stocky legs with cream-colored soles on each of i

In [7]:
# Helper functions
def _phrase_keep(s: str, vocab: set[str]) -> List[str]:
    # keep small noun/adjective chunks that include words from vocab
    kept = []
    # simple heuristic: keep 1–5 word spans around matches
    for m in vocab:
        if m in s.lower():
            # find windows around the match
            for mobj in re.finditer(re.escape(m), s, flags=re.I):
                start = max(0, mobj.start() - 25)
                end = min(len(s), mobj.end() + 25)
                frag = s[start:end]
                frag = re.sub(r"^[^A-Za-z]+|[^A-Za-z]+$", "", frag)
                kept.append(frag.strip(",;: "))
    return kept

def _drop_substrings(parts: List[str]) -> List[str]:
    keep=[]
    for i,p in enumerate(parts):
        pl=p.lower()
        if any(i!=j and pl in q.lower() for j,q in enumerate(parts)):
            continue
        keep.append(p)
    return keep

def _head_of(phrase: str) -> str:
    w = phrase.lower()
    for h in ANATOMY_WORDS:
        if h in w:
            return h
    # fallback: last word
    parts = re.findall(r"[a-z]+", w)
    return parts[-1] if parts else w
def _most_specific_by_head(phrases: List[str]) -> List[str]:
    best: Dict[str, str] = {}
    for p in phrases:
        p = re.sub(r"\s+", " ", p).strip(" ,.;")
        if not p: continue
        h = _head_of(p)
        if h not in best or len(p) > len(best[h]): best[h] = p
    return list(best.values())

def _unique_join(parts: List[str]) -> str:
    seen = set()
    out: List[str] = []
    for p in parts:
        p = re.sub(r"\s+", " ", p).strip()
        if not p: continue
        key = p.lower()
        if key not in seen:
            seen.add(key)
            out.append(p)
    # short, comma-separated
    return ", ".join(out)

# Clean up phrases so "long, blue ears" becomes "long blue ears"
# Build a single regex that matches: (adj|color)(, (adj|color))*  <head>
_HEADS = "s?|".join(sorted(ANATOMY_WORDS, key=len, reverse=True))+"s?"
_COLLAPSE_RE = re.compile(rf"\b({ADJ_COLOR_REGEX}(?:\s*,\s*{ADJ_COLOR_REGEX})*)\s+({_HEADS})\b", re.I)

def _collapse(text: str) -> str:
    if not text:
        return text
    def repl(m):
        # NOTE: avoid f-string with backslashes in the expression; build stepwise
        left = re.sub(r"\s*,\s*", " ", m.group(1))
        return left + " " + m.group(2)
    return _COLLAPSE_RE.sub(repl, text)

def _split(s: str) -> List[str]:
    return [p.strip() for p in (s or "").split(",") if p and p.strip()]

def _dedupe(parts: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for p in parts:
        key = p.lower()
        if key not in seen:
            seen.add(key); out.append(p)
    return out

In [8]:
# Do parsing related to biological description
# Extract phrases related to body, color and extra stuff.
def extract_body_phrases(sentences: List[str]) -> list[str]:
    # Strip leading “It has/It is” for cleaner matches
    text = ' '.join(sentences)
    t = re.sub(r"(?i)\bIt\s+(?:has|is)\s+", "", text)
    ph = [m.group(1).strip() for m in BODY_PHRASE_REGEX.finditer(t)]
    # Deduplicate by head, keep longest (most specific)
    by_head = {}
    for p in ph:
        head = next((h for h in ANATOMY_WORDS if re.search(rf"\b{h}\b", p, re.I)), p.split()[-1].lower())
        if head not in by_head or len(p) > len(by_head[head]):
            by_head[head] = p
    return list(by_head.values())


def extract_color_phrases(sentences: List[str]) -> list[str]:
    text = ' '.join(sentences)
    ph = [m.group(1).strip() for m in COLOR_PHRASE_REGEX.finditer(text)]
    # quick unique-preserving
    seen=set(); out=[]
    for p in ph:
        k=p.lower()
        if k not in seen: seen.add(k); out.append(p)
    return out


def extract_extras(sentences: List[str]) -> List[str]:
    res: List[str] = []
    for raw in sentences:
        s = _clean_sent(raw)
        # stripes/spots with modifiers retained
        for m in re.finditer(rf"\b({nonCapture(QUANT_DIR_REGEX)}\s+)?({nonCapture(COLOR_REGEX)}\s+)?(stripes?|spots?)\s+(?:on|along|across)\s+(its|the)\s+\w+", s, re.I):
            parts = [m.group(1) or "", m.group(2) or "", m.group(3), "on its back" if "back" in s.lower() else m.group(0).split(m.group(3),1)[-1].strip()]
            res.append(" ".join(p.strip() for p in parts if p).strip())
    return _uniq_keep(res)


# Start with apperance from bulbapedia and return fields for the POKEDESC construct
def appearance_to_pokedesc_fields(appearance: List[str]) -> Tuple[str, str, str]:
    """
    Returns (body, colors, extras) from a 1–2 sentence appearance description.
    """
    appearance_2sent =  ' '.join(appearance)
    text = _collapse(appearance_2sent or "")

    if not appearance:
        return "","","",""
    pat = r'(\w+)\s+(?=Pok[eé]mon)'
    form = re.search(pat, appearance[0])
    if form:
        form = form.group().strip()

    t_body = re.sub(r"(?i)\bIt\s+(?:has|is)\s+", "", text)
    # Collect phrases as "adj_run head" (with commas collapsed to spaces)
    body_phrases = []
    for m in BODY_PHRASE_REGEX.finditer(t_body):
        adj_run = (m.group(1) or "").strip()
        adj_run = re.sub(r"\s*,\s*", " ", adj_run)  # "long, pointed " -> "long pointed "
        head    = m.group(2)
        phrase  = (adj_run + " " + head).strip()
        # Skip bare head-only duplicates like "mouth" if you'll prefer modified variants later
        body_phrases.append((phrase, head.lower()))

    # Keep the longest phrase per head (most specific)
    best_by_head = {}
    for phrase, head in body_phrases:
        if head not in best_by_head or len(phrase) > len(best_by_head[head]):
            best_by_head[head] = phrase

    body_list = _drop_substrings(_dedupe(list(best_by_head.values())))

    # COLORS: <color> <target> (include tips/eyes/fur/stripes/spots/back)
    COLOR_RE = re.compile(rf"\b(({COLOR_REGEX})\s+({ANATOMY_REGEX}))\b", re.I)
    colors_list = _drop_substrings(_dedupe([m.group(1) for m in COLOR_RE.finditer(text)]))

    # EXTRAS: (two|horizontal|dark|light)? (color)? (stripes|spots) on (its|the) back
    EXTRAS_RE = re.compile(
        rf"\b({QUANT_DIR_REGEX}?\s*(?:{COLOR_WORDS}\s+)?"
        r"(?:stripes?|spots?)\s+(?:on|along|across)\s+(?:its|the)\s+back)\b", re.I)
    extras_list = _drop_substrings(_dedupe([m.group(1) for m in EXTRAS_RE.finditer(text)]))

    # Cross-field: if extras mentions stripes/spots/back, drop those heads from colors + substring overlaps
    if extras_list:
        colors_list = [p for p in colors_list if not any(k in p.lower() for k in ("stripes","spots","back"))]
        ex_low = [e.lower() for e in extras_list]
        colors_list = [p for p in colors_list if not any(p.lower() in e for e in ex_low)]

    return ", ".join(body_list), ", ".join(colors_list), ", ".join(extras_list)

# Clean up descriptive fields so there's no redundancy
def normalize_entry_fields(form: str, body: str, colors: str, extras: str, mood: str = "") -> Dict[str, str]:
    """Normalize one POKEDESC entry. Returns dict with clean body/colors/extras/mood."""
    # 1) Collapse adj/color runs
    body_c, colors_c, extras_c = (_collapse(body or ""),
                                  _collapse(colors or ""),
                                  _collapse(extras or ""))

    # 2) Split + basic dedupe
    body_list   = _drop_substrings(_dedupe(_split(body_c)))
    colors_list = _drop_substrings(_dedupe(_split(colors_c)))
    extras_list = _drop_substrings(_dedupe(_split(extras_c)))

    # 3) Keep most specific phrase per head within each field
    body_list   = _most_specific_by_head(body_list)
    colors_list = _most_specific_by_head(colors_list)
    extras_list = _most_specific_by_head(extras_list)

    # 4) Promote specificity across fields: if colors/extras have a longer phrase for same head, drop bare body one
    colors_heads = { _head_of(p): p for p in colors_list }
    extras_heads = { _head_of(p): p for p in extras_list }
    pruned_body = []
    for p in body_list:
        h = _head_of(p)
        pick = max([p, colors_heads.get(h, ""), extras_heads.get(h, "")], key=lambda s: len(s or ""))
        if pick == p:
            pruned_body.append(p)
    body_list = pruned_body

    # 5) Cross-field: extras outranks colors for pattern/location heads + substring cleanup
    if any((_head_of(p) in PATTERN_HEADS) or any(k in p.lower() for k in PATTERN_HEADS) for p in extras_list):
        # drop colors that duplicate pattern/location heads covered by extras
        colors_list = [p for p in colors_list if not any(k in p.lower() for k in PATTERN_HEADS)]

    # also drop color phrases that are substrings of any extra
    ex_low = [e.lower() for e in extras_list]
    colors_list = [p for p in colors_list if not any(p.lower() in e for e in ex_low)]

    # 6) Final tidy join
    def _join(parts: List[str]) -> str:
        parts = [re.sub(r"\s+", " ", p).strip(" ,.;") for p in parts if p]
        return ", ".join(_dedupe(parts))

    return {
        "form": form,
        "body":   _join(body_list),
        "colors": _join(colors_list),
        "extras": _join(extras_list),
        "mood":   re.sub(r"\s+", " ", mood or "").strip(" ,.;"),
    }

# Put field info into POKEDESC
def upsert_pokedesc(POKEDESC: Dict[str, Dict[str, str]], name: str, *, form="", body="", colors="", extras="", mood="") -> Dict[str, Dict[str, str]]:
    cur = POKEDESC.get(name, {})
    merged = {
        "form": form or cur.get("form",""),
        "body":   body   or cur.get("body",""),
        "colors": colors or cur.get("colors",""),
        "extras": extras or cur.get("extras",""),
        "mood":   mood   or cur.get("mood",""),
    }
    POKEDESC[name] = normalize_entry_fields(**merged)
    return POKEDESC

# Build prompt related to pokemon
def build_prompt(
    trigger: str,
    name: str,
    style: str = "3d",
    extra_tags: Optional[List[str]] = None,
    POKEDESC: Optional[Dict[str, Dict[str, str]]] = None,
) -> tuple[str, str]:
    d = (POKEDESC or {}).get(name, {})
    pos = [
        f"{trigger} pokemon",
        d.get("form", ""),
        d.get("body", ""),
        d.get("colors", ""),
        d.get("mood", ""),
        d.get("extras", ""),
        STYLE_PRESETS[style]["positive"],
    ]
    if extra_tags:
        pos += extra_tags
    # compact/clean
    positive = ", ".join(p for p in pos if p).replace("  ", " ").strip(", ")
    negative = STYLE_PRESETS[style]["negative"]
    return positive, negative


In [9]:
# Test cases to build fields in POKEDESC
if 1:
    POKEDESC: Dict[str, Dict[str, str]] = {}
    test_pokemon = "Pidgey"
    form, appearance = appearance_summary_from_bulbapedia(test_pokemon)
    (body, colors, extras) = appearance_to_pokedesc_fields(appearance)
    # Put features into pokedesc
    upsert_pokedesc(POKEDESC, test_pokemon, form=form,
                    body=body, colors=colors, extras=extras, mood="placid")
    pos, neg = build_prompt(trigger=f"<{test_pokemon}_token>", name=test_pokemon, style="3d", POKEDESC=POKEDESC)
    print(appearance)
    print(POKEDESC)
    print("POS:", pos)
    print("NEG:", neg)


['It has a short, stubby beak and feet with two toes in front and one in back.', 'Just under its crest are its narrow eyes, which have white sclera and pupil and black irises.']
{'Pidgey': {'form': 'avian', 'body': 'back, crest, eyes', 'colors': '', 'extras': 'two, one', 'mood': 'placid'}}
POS: <Pidgey_token> pokemon, avian, back, crest, eyes, placid, two, one, 3d render, studio lighting, high detail, global illumination, subsurface scattering
NEG: low quality, blurry, deformed, extra limbs, worst quality, jpeg artifacts


In [10]:
# Load POKEDESC for all pokemon
if CREATE_POKEDESC:
    POKEDESC: Dict[str, Dict[str, str]] = {}
    for pokemon in POKEMON:
        print(pokemon)
        form, appearance = appearance_summary_from_bulbapedia(pokemon)
        (body, colors, extras) = appearance_to_pokedesc_fields(appearance)
        # Put features into pokedesc
        upsert_pokedesc(POKEDESC, pokemon, form=form,
                        body=body, colors=colors, extras=extras, mood="placid")

    OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
    OUT_JSON.write_text(json.dumps(POKEDESC, indent=2), encoding="utf-8")
    print(f"Wrote auto POKEDESC → {OUT_JSON}")

else:
    POKEDESC = json.loads(OUT_JSON.read_text())

In [11]:
# Test - look at the descriptor for random pokemon
POKEDESC['charizard']

{'body': 'compact creature, stylized proportions',
 'colors': 'white, gray',
 'mood': 'mischievous',
 'extras': 'soft bounce light, gentle shadow on ground'}

### Create Pokemon image samples
To train Textual Inversion (TI) to images and terms foreign to the existing CLIP model, we need several samples of the creature for training. This also helps with building the PEFT adapter (LoRA) to learn the general pokemon form. Our process:</br>
<ul><li>Download images from Pokemon API</li>
<li>Augment images by shifting and adding noise to increase image samples</li>
</ul>

In [12]:
from torchvision import transforms
from local_tools import get_device
import os
from pathlib import Path
from dataclasses import dataclass
from PIL import Image
from pydantic import BaseModel
from pokemon_mtg_ref.pokemon_mtg import TYPE_TO_MTG_SUBTYPE, TYPE_TO_COLORS, ABILITY_MAP

POKEAPI_BASE = "https://pokeapi.co/api/v2"
POKE_ASSET_BASE = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other"
DATASET_PATH = Path("./datasets/pokemon-images")


@dataclass
class ImageRecord:
    url: str
    path: Path
    index: int


class PokemonCard(BaseModel):
    """Structured MTG-style card spec for rendering & prompting."""

    name: str
    mana_cost: str
    colors: List[str]
    type_line: str
    rarity: str
    rules_text: List[str]
    power: int
    toughness: int
    flavor_text: Optional[str] = None
    image_prompt: str
    negative_prompt: Optional[str] = None
    lora_trigger_token: Optional[str] = None

class PokemonInfo(BaseModel):
    """Minimal subset of PokeAPI info needed for prompts and MTG mapping."""

    name: str
    id: int
    types: List[str]
    abilities: List[str]
    is_legendary: bool
    is_mythical: bool
    height_decimeters: int
    weight_hectograms: int
    base_stats: Dict[str, int]
    flavor_text: Optional[str]


class PokeAPI:
    """Thin wrapper around PokeAPI endpoints."""

    SESSION = requests.Session()
    SESSION.headers.update({
        "User-Agent": "pokemon-mtg-lora-toolkit/1.1 (+https://example.local)",
        "Accept": "application/json, */*;q=0.1",
    })

    @staticmethod
    def _get(url: str) -> dict:
        r = PokeAPI.SESSION.get(url, timeout=25)
        r.raise_for_status()
        return r.json()

    @classmethod
    def fetch(cls, name: str) -> PokemonInfo:
        sname = name.strip().lower()
        pokemon = cls._get(f"{POKEAPI_BASE}/pokemon/{sname}")
        species = cls._get(pokemon["species"]["url"])

        types = [t["type"]["name"].capitalize() for t in pokemon["types"]]
        abilities = [a["ability"]["name"].replace("-", " ").title() for a in pokemon["abilities"]]

        # Prefer English flavor text, clean whitespace/newlines.
        flavor = None
        for entry in species.get("flavor_text_entries", []):
            if entry.get("language", {}).get("name") == "en":
                flavor = re.sub(r"\s+", " ", entry.get("flavor_text", "")).strip()
                break

        base_stats = {s["stat"]["name"]: s["base_stat"] for s in pokemon["stats"]}

        return PokemonInfo(
            name=pokemon["name"].capitalize(),
            id=pokemon["id"],
            types=types,
            abilities=abilities,
            is_legendary=species.get("is_legendary", False),
            is_mythical=species.get("is_mythical", False),
            height_decimeters=pokemon.get("height", 0),
            weight_hectograms=pokemon.get("weight", 0),
            base_stats=base_stats,
            flavor_text=flavor,
        )
    
def download_image_indexed(
    url: str,
    out_dir: Path,
    prefix: str,
    index: int,
    min_size: Tuple[int, int],
    max_retries: int = 3,
    upscale_if_small: bool = True,                
    min_upscale_src: Tuple[int, int] = (256, 256) 
) -> Optional[Path]:
    """Download and save as `<prefix>_<index>.jpg|png` (SVG->PNG if cairosvg).
    Upscales if below min_size (unless source is too tiny).
    """
    tries = 0
    while tries <= max_retries:
        try:
            resp = SESSION.get(url, timeout=30)
            resp.raise_for_status()
            data = resp.content

            # SVG -> PNG if available
            if url.lower().endswith(".svg"):
                if cairosvg is None:
                    return None
                tgt_w = max(min_size[0], 1024)
                tgt_h = max(min_size[1], 1024)
                data = cairosvg.svg2png(bytestring=data, output_width=tgt_w, output_height=tgt_h)

            im = Image.open(io.BytesIO(data))
            im = im.convert("RGBA") if _has_alpha(im) else im.convert("RGB")
            w, h = im.size

            # Enforce/repair min size via upscale
            if (w < min_size[0] or h < min_size[1]):
                if not upscale_if_small or w < min_upscale_src[0] or h < min_upscale_src[1]:
                    return None
                scale = max(min_size[0] / max(1, w), min_size[1] / max(1, h))
                new_w, new_h = int(round(w * scale)), int(round(h * scale))
                im = im.resize((new_w, new_h), Image.LANCZOS)

            ext = "png" if _has_alpha(im) else "jpg"
            dest = out_dir / f"{prefix}_{index}.{ext}"
            if dest.exists():
                k = index
                while dest.exists():
                    k += 1
                    dest = out_dir / f"{prefix}_{k}.{ext}"
                index = k

            if ext == "png":
                im.save(dest, format="PNG")
            else:
                im.save(dest, format="JPEG", quality=95, subsampling=1)
            return dest
        except Exception:
            if tries == max_retries:
                return None
            _sleep_backoff(tries)
            tries += 1
    return None


def sanitize_filename(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)

def collect_example_images(
    pokemon_name: str,
    pokemon_id: int,
    out_dir: Path,
    num_images: int,
    min_size: Tuple[int, int],
    image_provider: str = "multi",
    download_sleep: float = 0.8,
) -> List[ImageRecord]:
    out_images = out_dir / "images"
    out_images.mkdir(parents=True, exist_ok=True)
    manifest_path = out_images / "manifest.jsonl"
    cache_path = out_images / "url_cache.txt"

    records: List[ImageRecord] = []
    seen_urls: set[str] = _read_url_cache(cache_path)

    pokeart = PokeAPIOfficialArtProvider()
    wiki = WikimediaProvider()

    queries = [
        f"{pokemon_name} official art",
        f"{pokemon_name} illustration",
        f"{pokemon_name} concept art",
        f"{pokemon_name} fanart",
        f"{pokemon_name} trading card art",
    ]

    # Static fallbacks (often large)
    slug = pokemon_name.lower().replace("♀", "f").replace("♂", "m").replace(" ", "-")
    pdb_fallbacks = [
        f"https://img.pokemondb.net/artwork/large/{slug}.jpg",
        f"https://img.pokemondb.net/artwork/{slug}.jpg",
    ]

    prefix = _normalize_prefix(pokemon_name)
    next_idx = _next_index(prefix, out_images)

    def add_urls(urls: List[str]) -> None:
        nonlocal next_idx, seen_urls
        for u in urls:
            if len(records) >= num_images:
                break
            if u in seen_urls:
                continue
            path = download_image_indexed(u, out_images, prefix, next_idx, min_size=min_size)
            if path:
                records.append(ImageRecord(url=u, path=path, index=next_idx))
                _append_manifest(manifest_path, next_idx, path.name, u)
                seen_urls.add(u)
                next_idx += 1
                time.sleep(max(0.0, download_sleep))

    if image_provider in ("pokeapi", "multi") and len(records) < num_images:
        add_urls(pokeart.search(pokemon_id))

    if image_provider in ("wikimedia", "multi") and len(records) < num_images:
        add_urls(wiki.search(f"{pokemon_name} pokemon", max_results=max(12, num_images)))

    if len(records) < num_images:
        add_urls(pdb_fallbacks)

    _write_url_cache(cache_path, seen_urls)
    return records


def write_captions(records: List[ImageRecord], prompt: str, token: str) -> None:
    """Create per-image .txt caption files for common LoRA trainers.
    Why: Many LoRA scripts read `image.jpg` + `image.jpg.txt` to load prompts.
    """
    for rec in records:
        txt_path = rec.path.with_suffix(rec.path.suffix + ".txt")
        txt = f"{token} {prompt}"
        txt_path.write_text(txt)



In [13]:

def color_identity(types: List[str]) -> List[str]:
    ids = []
    for t in types:
        c = TYPE_TO_COLORS.get(t.lower(), "C")
        if c not in ids:
            ids.extend(c)
    return ids or ["C"]


def mtg_keywords(abilities: List[str]) -> List[str]:
    out: List[str] = []
    for a in abilities:
        k = ABILITY_MAP.get(a)
        if k and k not in out:
            out.append(k)
    return out

def scaled_stat(value: int, lo: int = 1, hi: int = 12, base: int = 50, spread: int = 100) -> int:
    """Map a Pokémon base stat (~0-255) to an MTG-ish P/T scale.

    Why: Keeps numbers small and printable while preserving relative differences.
    """
    x = (value - base) / spread
    y = (hi - lo) / 2 * x + (lo + hi) / 2
    return max(lo, min(hi, int(round(y))))


def save_card_and_prompts(card: PokemonCard, out_dir: Path) -> None:
    (out_dir / "meta").mkdir(parents=True, exist_ok=True)
    (out_dir / "prompts").mkdir(parents=True, exist_ok=True)

    (out_dir / "meta" / "card.json").write_text(json.dumps(card.model_dump(), indent=2))
    (out_dir / "prompts" / "positive.txt").write_text(card.image_prompt)
    if card.negative_prompt:
        (out_dir / "prompts" / "negative.txt").write_text(card.negative_prompt)

def build_card(p: PokemonInfo) -> PokemonCard:
    colors = color_identity(p.types)

    power = scaled_stat(p.base_stats.get("attack", 60))
    toughness = scaled_stat(p.base_stats.get("defense", 60))

    rarity = "Mythic Rare" if (p.is_mythical or p.is_legendary) else "Rare"

    subtype = TYPE_TO_MTG_SUBTYPE.get(p.types[0], "Beast") if p.types else "Beast"
    type_line = ("Legendary " if (p.is_mythical or p.is_legendary) else "") + f"Creature — {subtype}"

    kws = mtg_keywords(p.abilities)
    rules: List[str] = []
    if kws:
        rules.append(", ".join(sorted(set(kws))))
    if len(p.types) > 1:
        rules.append(f"{p.types[1]} Alignment — {p.name} gets +1/+1 until end of turn.")

    pos, neg = compose_mtg_art_prompt(p)
    mana = mana_from_stats(power, toughness, colors)

    return PokemonCard(
        name=p.name,
        mana_cost=mana,
        colors=colors,
        type_line=type_line,
        rarity=rarity,
        rules_text=rules or ["—"],
        power=power,
        toughness=toughness,
        flavor_text=p.flavor_text,
        image_prompt=pos,
        negative_prompt=neg,
        lora_trigger_token=f"<{p.name.lower()}-lora>",
    )

def emit_example_lora_script(out_dir: Path, train_steps: int = 1600) -> None:
    """Emit a simple LoRA training script for Diffusers users.

    Why: Provides a starting point; adjust as needed for your environment/GPU.
    """
    script = textwrap.dedent(
        f"""
        #!/usr/bin/env bash
        set -euo pipefail

        # Example SD 1.5 LoRA training with Hugging Face Diffusers
        # Adjust model, batch size, resolution, lr, and steps for your setup.

        MODEL_ID={MODEL_ID}
        TRAIN_DIR="{out_dir}/images"
        OUTPUT_DIR="{out_dir}/lora"

        accelerate launch \
          -m diffusers.examples.text_to_image.train_text_to_image_lora \
          --pretrained_model_name_or_path "$MODEL_ID" \
          --train_data_dir "$TRAIN_DIR" \
          --caption_column "text" \
          --resolution 768 \
          --train_batch_size 2 \
          --gradient_accumulation_steps 4 \
          --learning_rate 1e-4 \
          --lr_scheduler cosine \
          --lr_warmup_steps 100 \
          --max_train_steps {train_steps} \
          --validation_prompt "<your-token> Magic: the Gathering-style fantasy illustration" \
          --seed 42 \
          --output_dir "$OUTPUT_DIR"
        """
    ).strip()
    path = out_dir / "train_lora.sh"
    path.write_text(script + "\n")
    os.chmod(path, 0o755)

def prepare_lora_dataset(
    name: str,
    num_images: int,
    min_w: int,
    min_h: int,
    image_provider: str,
    download_sleep: float,
) -> None:
    out = DATASET_PATH / name.lower()
    out.mkdir(parents=True, exist_ok=True)

    p = PokeAPI.fetch(name)
    card = build_card(p)

    # Search 'image_provider' for an image of pokemon 'name'
    records = collect_example_images(
        p.name, p.id, out, num_images=num_images, min_size=(min_w, min_h), image_provider=image_provider, download_sleep=download_sleep
    )
    # Write descriptive captions for each image. e.g. if image is pokemon.png, the caption file is pokemon.png.txt.
    write_captions(records, prompt=card.image_prompt, token=card.lora_trigger_token or "<token>")
    # Create ./meta/card.json which holds info about the pokemon in MTG terms.
    # Create ./prompts/positive.txt and ./prompts/negative.txt as prompts to build MTG version of image using static diffusion. (positive & negative prompts)
    save_card_and_prompts(card, out)
    # Generate a shell script to create a LoRA of the image
    emit_example_lora_script(out)

    summary = {
        "pokemon": p.model_dump(),
        "card": card.model_dump(),
        "images_downloaded": len(records),
        "output_dir": str(out),
        "image_provider": image_provider,
        "filenames": [r.path.name for r in records],
    }
    return summary

In [14]:
class _Aug:
    """Light, identity-preserving augmentations. Avoid heavy warps.
    Why: inflate dataset diversity without drifting identity.
    """
    def __init__(
        self,
        resolution: int = 768,
        rotate_deg: float = 8.0,
        translate_pct: float = 0.06,
        scale_range: Tuple[float, float] = (0.95, 1.05),
        jitter: Tuple[float, float, float, float] = (0.08, 0.08, 0.08, 0.04),
        blur_sigma: Tuple[float, float] = (0.01, 0.8),
        hflip_prob: float = 0.5,
    ) -> None:
        self.pre = transforms.Compose([
            transforms.RandomHorizontalFlip(p=hflip_prob),
            transforms.RandomAffine(
                degrees=rotate_deg,
                translate=(translate_pct, translate_pct),
                scale=scale_range,
                interpolation=transforms.InterpolationMode.BILINEAR,
                fill=0,
            ),
            transforms.ColorJitter(*jitter),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=blur_sigma)], p=0.3),
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.CenterCrop(resolution),
        ])

    def __call__(self, im: Image.Image) -> Image.Image:
        if im.mode != "RGB":
            im = im.convert("RGB")
        return self.pre(im)


def _scan_images(img_dir: Path) -> List[Path]:
    return sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])


def _copy_caption(src_img: Path, dst_img: Path, default_caption: Optional[str] = None) -> None:
    cap_src = src_img.with_suffix(src_img.suffix + ".txt")
    cap_dst = dst_img.with_suffix(dst_img.suffix + ".txt")
    if cap_src.exists():
        cap_dst.write_text(cap_src.read_text(encoding="utf-8"), encoding="utf-8")
    elif default_caption:
        cap_dst.write_text(default_caption, encoding="utf-8")


def augment_images_to_count(
    pokemon_dir: Path,
    target_count: int = 20,
    max_per_source: int = 4,
    resolution: int = 768,
    seed: int = 123,
    output_format: str = "jpg",
) -> Dict:
    """Materialize augmented images in-place until image count >= target_count.
    Why: Keeps LoRA datasets large enough without manual scraping.
    """
    rng = random.Random(seed)

    img_dir = pokemon_dir / "images"
    meta_card = pokemon_dir / "meta" / "card.json"

    img_dir.mkdir(parents=True, exist_ok=True)
    files = _scan_images(img_dir)

    default_caption = None
    if meta_card.exists():
        try:
            card = json.loads(meta_card.read_text())
            token = card.get("lora_trigger_token") or "<token>"
            default_caption = f"{token} creature, fantasy monster"
        except Exception:
            default_caption = None

    # Determine prefix from folder name
    prefix = _normalize_prefix(pokemon_dir.name)

    # Early exit if already sufficient
    start_count = len(files)
    if start_count >= target_count:
        return {"pokemon": pokemon_dir.name, "start": start_count, "end": start_count, "made": 0}

    aug = _Aug(resolution=resolution)

    # Cycle through originals producing up to max_per_source each pass
    made = 0
    i = 0
    originals = list(files)
    if not originals:
        return {"pokemon": pokemon_dir.name, "start": 0, "end": 0, "made": 0, "note": "no originals"}

    next_idx = _next_index(prefix, img_dir)

    while (start_count + made) < target_count:
        src = originals[i % len(originals)]
        i += 1
        # generate variants from this source
        for _ in range(max_per_source):
            if (start_count + made) >= target_count:
                break
            try:
                im = Image.open(src)
                aug_im = aug(im)
                ext = ".jpg" if output_format.lower() == "jpg" else ".png"
                out = img_dir / f"{prefix}_{next_idx}{ext}"
                if out.suffix.lower() == ".png":
                    aug_im.save(out, format="PNG")
                else:
                    aug_im.save(out, format="JPEG", quality=95, subsampling=1)
                _copy_caption(src, out, default_caption=default_caption)
                made += 1
                next_idx += 1
            except Exception:
                continue

    end = start_count + made
    return {"pokemon": pokemon_dir.name, "start": start_count, "end": end, "made": made}


def augment_all_pokemon(
    dataset_root: Path | str = DATASET_PATH,
    target_count: int = 20,
    max_per_source: int = 4,
    resolution: int = 768,
    seed: int = 123,
    output_format: str = "jpg",
) -> List[Dict]:
    """Run augmentation across all Pokémon subfolders under `dataset_root`.
    Why: Batch-scale preparation for LoRA training.
    """
    root = Path(dataset_root)
    reports: List[Dict] = []
    for poke_dir in sorted(root.iterdir()):
        if not poke_dir.is_dir():
            continue
        if not (poke_dir / "images").exists():
            continue
        rep = augment_images_to_count(
            pokemon_dir=poke_dir,
            target_count=target_count,
            max_per_source=max_per_source,
            resolution=resolution,
            seed=seed,
            output_format=output_format,
        )
        reports.append(rep)
    # Simple printout
    total_made = sum(r.get("made", 0) for r in reports)
    print(f"Augmented {len(reports)} pokemon folders; new images created: {total_made}")
    return reports


In [15]:
# Generate sample images for each of the "top pokemon"
# Map from base pokemon name to form found in pokeapi
POKE_MAP = {'deoxys':'deoxys-normal', 'giratina':'giratina-origin', 'tapu koko':'tapu-koko', 'lycanroc':'lycanroc-midday',
            "sirfetch'd":"sirfetchd", 'aegislash':'aegislash-shield', 'mimikyu':'mimikyu-disguised'}

if CREATE_DATASET:
    for pokemon in POKEMON:
        pokemon = pokemon.lower()
        if pokemon in POKE_MAP.keys():
            pokemon = POKE_MAP[pokemon]
        summary = prepare_lora_dataset(
            name=pokemon,
            num_images=10,
            min_w=256, min_h=256,
            image_provider="pokeapi",
            download_sleep=0.8,
        )
        print(pokemon)
    # Create noisy/shifted versions of pokemon so there are more samples for the LoRA action.
    augment_all_pokemon()

#### Audit the image folders to see if the data is reasonable
We want ~20 images per pokemon and a small portion of them under 512 pixels

In [16]:

# Initialize
DEVICE = get_device()
if DEVICE == 'mps':
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
ROOT = Path("./datasets/pokemon-images")

def audit_dataset(root: Path = ROOT, min_ok: int = 20, show=10):
    rows = []
    for d in sorted(root.iterdir()):
        if not d.is_dir(): 
            continue
        imgs = sorted((d/"images").glob("*.[jp][pn]g"))
        n = len(imgs)
        caps_missing = 0
        small = 0
        for p in imgs[:200]:  # sample first 200 to keep it quick
            if not (p.with_suffix(p.suffix + ".txt")).exists():
                caps_missing += 1
            try:
                with Image.open(p) as im:
                    w, h = im.size
                    if w < 512 or h < 512:
                        small += 1
            except Exception:
                pass
        rows.append((d.name, n, caps_missing, small))
    rows.sort(key=lambda r: r[1])
    print(f"Pokémon folders audited: {len(rows)}")
    print(f"{'name':<20}  {'imgs':>4}  {'missing_caps':>12}  {'<512px':>7}")
    for r in rows[:show]:  # show worst 10 by count
        print(f"{r[0]:<20}  {r[1]:>4}  {r[2]:>12}  {r[3]:>7}")
    low = [r for r in rows if r[1] < min_ok]
    if low:
        print(f"\n{len(low)} folders below {min_ok} images. Consider scraping/augmenting more.")
    return rows

_ = audit_dataset()


Pokémon folders audited: 109
name                  imgs  missing_caps   <512px
absol                   20             0        0
aegislash-blade         20             0        2
aegislash-shield        20             0        3
alakazam                20             0        3
amaura                  20             0        2
ampharos                20             0        0
annihilape              20             0        3
arcanine                20             0        2
arceus                  20             0        2
articuno                20             0        2


### Adapt established CLIP model to Pokemon images and token names
#### Load CLIP model

In [17]:
# # SD v1.5 LoRA — Minimal, Modular, MPS‑safe (Cells 1–3)
# Cell 1 = Load models; Cell 2 = Dataset/Loader; Cell 3 = Trainer (PEFT LoRA fallback).

# %%
# Load models (no training yet)
from __future__ import annotations
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
from local_tools import get_device
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, PeftModel, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

ADAPTER_DIR = Path("./results/pokemon-images/lora_peft_minimal")  # change if needed
OUT_DIR = Path("./results/pokemon-images/samples"); OUT_DIR.mkdir(parents=True, exist_ok=True)
VECTORS_DIR = Path("./results/pokemon-images/ti_embeddings")

MODEL_ID="runwayml/stable-diffusion-v1-5"
BUILD_LORA = False
EMBED_TI = False # Perform token embeddings
OVERWRITE_EXISTING_TOKENS = False # Overwrite existing tokens (if EMBED_TI is true, then only overwrite the new tokens)


# Prefer a task type that won't add generation mixins to UNet
SAFE_TASK = getattr(TaskType, "FEATURE_EXTRACTION", getattr(TaskType, "SEQ_2_SEQ_LM"))


# Configure dataset path here
OUTDIR = Path("./results/pokemon-images/lora_peft_minimal")
OUTDIR.mkdir(parents=True, exist_ok=True)
EXAMPLE_PATH = Path("./results/stable_diffusion")
CACHE_LATENTS = True
DEVICE = get_device()
print("Device:", DEVICE)
if DEVICE == 'mps':
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Whenever we add tokens to the LLM, we need to rebuild the  token ID -> vector lookup table. MPS does not do this smoothly.
# So if on MPS, need to move the encoder to CPU, then resize the lookup table, then move back to device
def add_tokens(tokenizer, text_enc, new_tokens, device):
    # Add tokens to tokenizer
    added = tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})  # may be 0
    if not added:
        return  # nothing to do
    
    # Move encoder to CPU on MPS before resize
    if device.type == "mps":
        text_enc.to("cpu")

    # Resize embeddings (Parameter gets replaced here)
    text_enc.resize_token_embeddings(len(tokenizer))

    # 5) Move back to target device
    if device.type == "mps":
        text_enc.to(device)


vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
text_tok = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_enc = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")


vae.to(DEVICE).eval()
text_enc.to(DEVICE).eval()
unet.to(DEVICE).train(False) # will re‑enable grads only for LoRA in Cell 3


print("Loaded:", type(vae).__name__, type(text_enc).__name__, type(unet).__name__)


# Quick smoke forward through UNet (random latents + empty text) to verify kernels
with torch.no_grad():
    lat = torch.randn(1, 4, 64, 64, device=DEVICE)
    t = torch.randint(0, scheduler.config.num_train_timesteps, (1,), device=DEVICE)
    ctx = torch.zeros(1, 77, unet.config.cross_attention_dim, device=DEVICE)
    out = unet(lat, t, encoder_hidden_states=ctx)
print("UNet forward OK; sample shape:", out.sample.shape)

Device: mps


/opt/anaconda3/envs/ai/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loaded: AutoencoderKL CLIPTextModel UNet2DConditionModel
UNet forward OK; sample shape: torch.Size([1, 4, 64, 64])


In [18]:
class CaptionImageDataset(Dataset):
    def __init__(self, root: str | Path, size: int = 512, center_crop: bool = True,
                 cache_latents: bool = False,
                 vae: Optional[AutoencoderKL] = None,
                 device: Optional[torch.device] = None):
        self.root = Path(root)
        self.size = size
        self.center_crop = center_crop
        self.cache_latents = cache_latents
        self.vae = vae
        self.device = device
        self.items: List[tuple[Path, str]] = []
        meta = self.root / "captions.txt"
        if meta.exists():
            for line in meta.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                p, cap = line.split("\t", 1)
                self.items.append((self.root / p, cap))
        else:
            for p in sorted(self.root.glob("**/*")):
                if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}: continue
                txt = p.with_suffix(".txt")
                cap = txt.read_text(encoding="utf-8").strip() if txt.exists() else ""
                self.items.append((p, cap))
        if not self.items:
            raise FileNotFoundError(f"No data under {self.root}")
        self.tf = T.Compose([
            T.Resize(self.size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(self.size) if self.center_crop else T.RandomCrop(self.size),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.latent_cache: Dict[int, torch.Tensor] = {}

    def __len__(self): return len(self.items)

    @torch.no_grad()
    def _encode_latent(self, img: torch.Tensor) -> torch.Tensor:
        assert self.vae is not None and self.device is not None
        img = img.unsqueeze(0).to(self.device)
        latent = self.vae.encode(img).latent_dist.sample() * 0.18215
        return latent.squeeze(0).cpu()

    def __getitem__(self, idx: int):
        path, cap = self.items[idx]
        img = Image.open(path).convert("RGB")
        img = self.tf(img)
        if self.cache_latents:
            if idx not in self.latent_cache:
                self.latent_cache[idx] = self._encode_latent(img)
            return {"latent": self.latent_cache[idx], "caption": cap}
        return {"pixel_values": img, "caption": cap}


train_ds = CaptionImageDataset(DATASET_PATH, size=512, center_crop=True,
                cache_latents=CACHE_LATENTS, vae=vae if CACHE_LATENTS else None, device=DEVICE)
print("Dataset size:", len(train_ds))

# MPS/CPU-safe loader: no multiprocessing/shared memory
num_workers = 0 if DEVICE.type in ("mps", "cpu") else 4

loader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=num_workers,
    prefetch_factor=None if num_workers == 0 else 2,
    persistent_workers=False if num_workers == 0 else True,
    pin_memory=False,   # keep False on MPS/CPU
    drop_last=True,
)

# sanity: fetch one batch
batch = next(iter(loader))
x = batch["latent"] if CACHE_LATENTS else batch["pixel_values"]
print("One batch OK;", "latent" if CACHE_LATENTS else "pixels", x.shape, "| captions:", len(batch["caption"]))



Dataset size: 2182
One batch OK; latent torch.Size([2, 4, 64, 64]) | captions: 2


#### Build PEFT LoRA to establish the Pokemon visual manifold

In [19]:
# Trainer with PEFT LoRA fallback (version-safe TaskType)
# Hyperparams for the run 
LR, WD, STEPS, RANK = 1e-4, 1e-2, 10000, 8


@torch.no_grad()
def tokenize_captions(tokenizer: CLIPTokenizer, caps: List[str], device: torch.device, max_length: int = 77) -> torch.Tensor:
    return tokenizer(caps, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt").input_ids.to(device)

def attach_lora(unet: UNet2DConditionModel, rank: int = 8, alpha: Optional[int] = None) -> Tuple[torch.nn.Module, str]:
    """PEFT LoRA on UNet attention linears. Returns (wrapped_unet, mode)."""
    cfg = LoraConfig(
        r=rank,
        lora_alpha=(alpha or rank),
        lora_dropout=0.0,
        bias="none",
        task_type=SAFE_TASK,          # <-- version-safe fallback
        target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    )
    wrapped = get_peft_model(unet, cfg)
    return wrapped, "peft"


# Attach LoRA
unet.train(False)  # freeze base
unet_lora, mode = attach_lora(unet, rank=RANK)
print("LoRA mode:", mode)

# Use the base UNet (with LoRA applied) for forward to avoid wrapper expecting input_ids
core_unet = getattr(unet_lora, "base_model", getattr(unet_lora, "model", unet_lora))

# Collect only LoRA params
trainable = [p for n, p in unet_lora.named_parameters() if "lora_" in n]
for n, p in unet_lora.named_parameters():
    p.requires_grad_("lora_" in n)
print("LoRA tensors:", len(trainable), "| total params:", sum(p.numel() for p in trainable))

opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=WD)

if BUILD_LORA:
    SAVE_EVERY = 1000
    LOG_EVERY  = 200
    # Build a "style LoRA", so only train the UNet, not the text.
    text_enc.train(False)
    core_unet.train(True)

    global_step, epoch = 0, 0
    while global_step < STEPS:
        epoch += 1
        for batch in loader:
            if global_step >= STEPS:
                break

            # ---- forward ----
            latents = (batch["latent"] if CACHE_LATENTS else
                    (vae.encode(batch["pixel_values"].to(DEVICE)).latent_dist.sample() * 0.18215))
            latents = latents.to(DEVICE)

            noise = torch.randn_like(latents)
            t = torch.randint(0, scheduler.config.num_train_timesteps, (latents.size(0),),
                            device=DEVICE, dtype=torch.long)
            noisy = scheduler.add_noise(latents, noise, t)

            caps = batch["caption"] if isinstance(batch["caption"], list) else [batch["caption"]]
            with torch.no_grad():  # <- critical: avoid gradients through TE
                input_ids = tokenize_captions(text_tok, caps, DEVICE)
                ctx = text_enc(input_ids).last_hidden_state

            pred = core_unet(noisy, t, encoder_hidden_states=ctx).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())

            # ---- backward ----
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            opt.step()

            global_step += 1

            if global_step % LOG_EVERY == 0:
                print(f"epoch {epoch} | step {global_step:05d} | loss {loss.item():.4f}")

            if global_step % SAVE_EVERY == 0:
                ckpt_dir = OUTDIR / f"step_{global_step}"
                unet_lora.save_pretrained(str(ckpt_dir))
                print("checkpoint →", ckpt_dir)

    # final save
    unet_lora.save_pretrained(str(OUTDIR))
    print("Saved PEFT adapter →", OUTDIR)



LoRA mode: peft
LoRA tensors: 256 | total params: 1594368


#### Perform Pokemon token embeddings

In [20]:
# # SD v1.5 — Hybrid: Global LoRA + Per‑Character Textual Inversion
# Cells TI‑1..TI‑3: add special tokens (one per Pokémon dir) and train token embeddings.
# Minimal, MPS‑safe, extendable to ~100 tokens.

# %%
# TI‑1 — Discover concepts, add tokens, init embeddings
from __future__ import annotations
import json

import torch
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel
from local_tools import get_device
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # must be set before importing torch

vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
text_tok = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_enc = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# Helper function -- 
vae.to(DEVICE).eval()
text_enc.to(DEVICE).eval()
unet.to(DEVICE).train(False) # will re‑enable grads only for LoRA in Cell 3


# Config — point to the parent directory containing one subfolder per Pokémon
CONCEPTS_ROOT = Path("./datasets/pokemon-images")  # e.g., ./datasets/pokemon-images/gengar, ./.../pikachu, ...
TOKEN_PREFIX = "<pk_"     # tokens become like <pk_gengar>
TOKEN_SUFFIX = ">"
CLASS_WORD   = "pokemon"  # keeps style anchor consistent


# Discover concept folders
concept_dirs = sorted([p for p in CONCEPTS_ROOT.iterdir() if p.is_dir()])
concept_names = [p.name for p in concept_dirs]
print(f"Found {len(concept_names)} concepts:", concept_names[:10], ("..." if len(concept_names)>10 else ""))

# Build tokens
concept_to_token: Dict[str, str] = {name: f"{TOKEN_PREFIX}{name}{TOKEN_SUFFIX}" for name in concept_names}
SPECIAL_TOKENS: List[str] = list(concept_to_token.values())

# Add tokens to tokenizer
add_tokens(text_tok, text_enc, [t for t in SPECIAL_TOKENS if t not in text_tok.get_vocab()], DEVICE)

# Init each new token embedding from the class word vector (+ tiny noise)
base_id = text_tok.convert_tokens_to_ids(CLASS_WORD)
emb = text_enc.get_input_embeddings()
with torch.no_grad():
    base_vec = emb.weight[base_id].detach().clone()
    for name, tok in concept_to_token.items():
        tid = text_tok.convert_tokens_to_ids(tok)
        if tid is None or tid < 0:
            raise RuntimeError(f"Token not found after add_tokens: {tok}")
        # Only initialize if it looks fresh (heuristic: norm small)
        if emb.weight[tid].norm().item() < 1e-6 or True:
            emb.weight[tid].copy_(base_vec + 0.01 * torch.randn_like(base_vec))

# Persist mapping for later (inference, bookkeeping)
MAP_PATH = CONCEPTS_ROOT / "_ti_tokens.json"
MAP_PATH.write_text(json.dumps(concept_to_token, indent=2), encoding="utf-8")
print("Saved token map →", MAP_PATH)




Found 109 concepts: ['absol', 'aegislash-blade', 'aegislash-shield', 'alakazam', 'amaura', 'ampharos', 'annihilape', 'arcanine', 'arceus', 'articuno'] ...
Saved token map → datasets/pokemon-images/_ti_tokens.json


In [21]:

# %%
# TI‑2 — Dataset for a single concept (auto‑caption with its trigger token)
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

class TIConceptDataset(Dataset):
    def __init__(self, folder: Path, token: str, size: int = 512, center_crop: bool = True,
                 cache_latents: bool = True, vae: AutoencoderKL | None = None, device: torch.device | None = None,
                 class_word: str = CLASS_WORD):
        self.paths = sorted([p for p in folder.glob("**/*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}])
        if not self.paths:
            raise FileNotFoundError(f"No images under {folder}")
        self.caption = f"{token} {class_word}"
        self.cache_latents = cache_latents
        self.vae = vae
        self.device = device
        self.tf = T.Compose([
            T.Resize(size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(size) if center_crop else T.RandomCrop(size),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.latents: Dict[int, torch.Tensor] = {}

    def __len__(self): return len(self.paths)

    @torch.no_grad()
    def _encode_latent(self, img: torch.Tensor) -> torch.Tensor:
        assert self.vae is not None and self.device is not None
        img = img.unsqueeze(0).to(self.device)
        lat = self.vae.encode(img).latent_dist.sample() * 0.18215
        return lat.squeeze(0).cpu()

    def __getitem__(self, i: int):
        img = Image.open(self.paths[i]).convert("RGB")
        px = self.tf(img)
        if self.cache_latents:
            if i not in self.latents:
                self.latents[i] = self._encode_latent(px)
            return {"latent": self.latents[i], "caption": self.caption}
        return {"pixel_values": px, "caption": self.caption}


def build_loader_mps_safe(ds: Dataset, batch_size: int = 2) -> DataLoader:
    num_workers = 0 if DEVICE.type in ("mps","cpu") else 4
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        prefetch_factor=None if num_workers==0 else 2,
        persistent_workers=False if num_workers==0 else True,
        pin_memory=False,
        drop_last=True,
    )

print("TI dataset scaffolding ready.")

# %%
# TI‑3 — Train token embeddings (one concept at a time). UNet stays LoRA‑ready but frozen.
# --- 1) Replace train_one_token signature & internals to avoid globals ---
def train_one_token(
    concept_name: str,
    token: str,                       # <-- pass the token in
    steps: int = 1000,
    batch_size: int = 2,
    lr: float = 5e-3,
    cache_latents: bool = True,
) -> None:
    # sanity: token must exist as a single ID
    ids = text_tok(token, add_special_tokens=False).input_ids
    assert len(ids) == 1, f"Placeholder must be a single token: {token} -> {ids}"
    tid = ids[0]

    folder = CONCEPTS_ROOT / concept_name
    ds = TIConceptDataset(
        folder, token, size=512, center_crop=True,
        cache_latents=cache_latents, vae=vae if cache_latents else None, device=DEVICE
    )
    loader = build_loader_mps_safe(ds, batch_size=batch_size)

    # freeze UNet; put text encoder in train but freeze its params first
    vae.eval(); unet.eval()
    for p in unet.parameters(): p.requires_grad_(False)

    text_enc.train(True)
    for p in text_enc.parameters(): p.requires_grad_(False)  # freeze all

    # NOW unfreeze only the embedding Parameter and build optimizer on it
    emb_mod   = text_enc.get_input_embeddings()      # nn.Embedding
    emb_param = emb_mod.weight                       # nn.Parameter
    emb_param.requires_grad_(True)                   # <-- critical

    opt = torch.optim.AdamW([emb_param], lr=lr, weight_decay=0.0)
    # target embedding Parameter (optimize the Parameter itself)
    emb_mod = text_enc.get_input_embeddings()          # nn.Embedding
    assert isinstance(emb_mod.weight, torch.nn.Parameter)
    emb_param = emb_mod.weight                         # Parameter [V, H]
    assert emb_param.requires_grad, "Embedding weight must require grad"

    # optimizer on the embedding weight Parameter
    opt = torch.optim.AdamW([emb_param], lr=lr, weight_decay=0.0)

    # grad mask for single row
    mask = torch.zeros_like(emb_param, dtype=torch.bool); mask[tid] = True

    print(f"Training token {token} for {steps} steps on {len(ds)} imgs…")
    step = 0
    while step < steps:
        for batch in loader:
            if step >= steps: break

            # latents
            if cache_latents:
                latents = batch["latent"].to(DEVICE)
            else:
                latents = vae.encode(batch["pixel_values"].to(DEVICE)).latent_dist.sample() * 0.18215

            noise = torch.randn_like(latents)
            t = torch.randint(0, scheduler.config.num_train_timesteps, (latents.size(0),), device=DEVICE, dtype=torch.long)
            noisy = scheduler.add_noise(latents, noise, t)

            # captions must include the token
            caps = batch["caption"]
            if isinstance(caps, list):
                assert any(token in c for c in caps), f"Caption missing token {token} for {concept_name}"
            else:
                assert token in caps, f"Caption missing token {token} for {concept_name}"

            ids = text_tok(caps, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(DEVICE)
            ctx = text_enc(ids).last_hidden_state  # grads flow to emb weight

            pred = unet(noisy, t, encoder_hidden_states=ctx).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())

            opt.zero_grad(set_to_none=True)
            loss.backward()

            # keep grads on the single row
            if emb_param.grad is not None:
                emb_param.grad[~mask] = 0

            torch.nn.utils.clip_grad_norm_([emb_param], 1.0)
            opt.step()

            step += 1
            if step % 50 == 0:
                print(f"{concept_name:>16s} | step {step:05d} | loss {loss.item():.4f}")

    # save vector
    vec = emb_param.detach().cpu()[tid]
    out_dir = Path("./results/pokemon-images/ti_embeddings"); out_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"token": token, "vector": vec}, out_dir / f"{concept_name}.pt")
    print(f"Saved TI vector → {out_dir/concept_name}.pt")




print("TI training helpers ready. Example usage:")
print("train_one_token('gengar', steps=1000, batch_size=2, lr=5e-3)")

# %%
# TI-4 — Batch trainer: loop all concepts and train TI embeddings
from time import perf_counter

def estimate_steps(n_images: int, steps_per_image: int = 40, *, min_steps: int = 600, max_steps: int = 2000) -> int:
    """Rough heuristic: more images → more steps; clamp to sane bounds."""
    return int(max(min_steps, min(max_steps, n_images * steps_per_image)))

# --- 2) Wire mapping through train_all_tokens & improve error logs ---
def preflight_placeholders_for(names, *, token_map_path: Path) -> list[str]:
    """Add missing placeholders once, resize text encoder, assert shape match."""
    concept_to_token = json.loads(token_map_path.read_text())
    # targets: all if names falsy
    all_dirs_map = {p.name: p for p in concept_dirs}
    targets = list(all_dirs_map.keys()) if not names else [n for n in names if n in all_dirs_map]
    missing_dirs = [n for n in (names or []) if n not in all_dirs_map]
    if missing_dirs:
        print("[warn] missing concept dirs:", ", ".join(missing_dirs))

    placeholders = []
    for n in targets:
        tok = concept_to_token.get(n)
        if not tok:
            raise ValueError(f"Missing placeholder for: {n}")
        if "-" in tok:
            raise ValueError(f"{n}: placeholder '{tok}' has hyphen; use underscores.")
        if not (tok.startswith("<") and tok.endswith(">")):
            raise ValueError(f"{n}: placeholder '{tok}' must be wrapped like <name>.")
        placeholders.append(tok)

    # Determine which need adding (not single-id yet)
    add_tokens(text_tok, text_enc, [t for t in placeholders if len(text_tok(t, add_special_tokens=False).input_ids) != 1], DEVICE)


    # Final sanity: vocab size must equal embedding rows
    emb = text_enc.get_input_embeddings()
    vocab, rows = len(text_tok), emb.weight.shape[0]
    assert vocab == rows, f"mismatch: tokenizer {vocab} vs emb rows {rows}"
    # Also verify single-id
    for t in placeholders:
        ids = text_tok(t, add_special_tokens=False).input_ids
        assert len(ids) == 1, f"Still splits: {t} -> {ids}"

    # Optional: protect against fast-tokenizer splitting
    # text_tok.unique_no_split_tokens = list(set(text_tok.unique_no_split_tokens + placeholders))

    return targets

# ---- Use this at the START of train_selected_tokens ----
def train_selected_tokens(
    names=None, *, steps_per_image=40, min_steps=600, max_steps=1600, batch_size=2, lr=5e-3,
    cache_latents=True, overwrite_existing=False, token_map_path: Path | None = None,
):
    token_map_path = token_map_path or (CONCEPTS_ROOT / "_ti_tokens.json")

    # Preflight ONCE for this run
    targets = preflight_placeholders_for(names, token_map_path=token_map_path)

    # Loop and train
    out_dir = Path("./results/pokemon-images/ti_embeddings"); out_dir.mkdir(parents=True, exist_ok=True)
    concept_to_token = json.loads(token_map_path.read_text())

    ok, fail = 0, []
    for name in targets:
        vec_path = out_dir / f"{name}.pt"
        if vec_path.exists() and not overwrite_existing:
            print(f"[skip] {name}: {vec_path.name} exists"); ok += 1; continue

        folder = next((p for p in concept_dirs if p.name == name), None)
        if not folder:
            print(f"[warn] {name}: no dir; skipping"); continue
        n_imgs = sum(1 for p in folder.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"})
        if n_imgs == 0:
            print(f"[warn] {name}: no images; skipping"); continue

        steps = int(max(min_steps, min(max_steps, n_imgs * steps_per_image)))
        token = concept_to_token[name]

        # Recheck the critical invariant right before training
        emb = text_enc.get_input_embeddings()
        vocab, rows = len(text_tok), emb.weight.shape[0]
        assert vocab == rows, f"pre-train mismatch: tokenizer {vocab} vs emb rows {rows}"
        assert len(text_tok(token, add_special_tokens=False).input_ids) == 1

        print(f"[run ] {name}: {n_imgs} imgs → {steps} steps | token {token}")
        try:
            train_one_token(name, token=token, steps=steps, batch_size=batch_size, lr=lr, cache_latents=cache_latents)
            ok += 1
        except Exception as e:
            fail.append((name, f"{e.__class__.__name__}: {e}"))

    print(f"Selected TI done: {ok}/{len(targets)} ok")
    if fail:
        print("Errors:"); [print(" -", n, "→", msg) for n, msg in fail]

def train_all_tokens(
    steps_per_image: int = 40,
    min_steps: int = 600,
    max_steps: int = 2000,
    batch_size: int = 2,
    lr: float = 5e-3,
    overwrite_existing: bool = False,
    only: list[str] | None = None,
):
    start = perf_counter()
    done = 0
    errors: list[tuple[str, str]] = []

    # load mapping and normalize placeholders (avoid hyphens)
    token_map_path = CONCEPTS_ROOT / "_ti_tokens.json"
    concept_to_token_local: Dict[str, str] = json.loads(token_map_path.read_text())
    concept_to_token_local = { k:v.replace('-','_') for k,v in concept_to_token_local.items()}
    print(concept_to_token_local)
    # sanity: placeholders should be single-token style
    for k,v in concept_to_token_local.items():
        assert v.startswith("<") and v.endswith(">"), f"{k} → placeholder should be like <foo_bar>, got {v}"
        assert "-" not in v, f"{k} → avoid hyphens in token {v}; use underscores"

    targets = [p.name for p in concept_dirs]
    if only:
        only_set = set(only)
        targets = [t for t in targets if t in only_set]

    out_dir = Path("./results/pokemon-images/ti_embeddings"); out_dir.mkdir(parents=True, exist_ok=True)

    for name in targets:
        vec_path = out_dir / f"{name}.pt"
        if vec_path.exists() and not overwrite_existing:
            print(f"[skip] {name}: {vec_path.name} exists")
            done += 1
            continue

        folder = CONCEPTS_ROOT / name
        n_imgs = len([p for p in folder.glob("**/*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}])
        if n_imgs == 0:
            print(f"[warn] {name}: no images; skipping")
            continue

        token = concept_to_token_local.get(name)
        if not token:
            errors.append((name, "no placeholder token in _ti_tokens.json"))
            print(f"[fail] {name}: no placeholder token in map")
            continue

        steps = estimate_steps(n_imgs, steps_per_image, min_steps=min_steps, max_steps=max_steps)
        print(f"[run ] {name}: {n_imgs} imgs → {steps} steps (token {token})")
        try:
            train_one_token(name, token=token, steps=steps, batch_size=batch_size, lr=lr, cache_latents=True)
            done += 1
        except Exception as e:
            tb = traceback.format_exc(limit=2)
            errors.append((name, f"{e.__class__.__name__}: {e} | {tb.strip()}"))
            print(f"[fail] {name}: {e}")

    dt = perf_counter() - start
    print(f"Batch TI done: {done}/{len(targets)} ok in {dt/60:.1f} min")
    if errors:
        print("Errors:")
        for n, msg in errors:
            print(" -", n, "→", msg)


print("Batch trainer ready. Example:")
print("train_all_tokens(steps_per_image=40, min_steps=800, max_steps=1600, batch_size=2, lr=5e-3)")


TI dataset scaffolding ready.
TI training helpers ready. Example usage:
train_one_token('gengar', steps=1000, batch_size=2, lr=5e-3)
Batch trainer ready. Example:
train_all_tokens(steps_per_image=40, min_steps=800, max_steps=1600, batch_size=2, lr=5e-3)


In [ ]:
# Train all remaining tokens; skips ones already saved in ti_embeddings/
def estimate_steps(n_images: int, steps_per_image: int = 40, *, min_steps: int = 600, max_steps: int = 2000) -> int:
    return int(max(min_steps, min(max_steps, n_images * steps_per_image)))

if EMBED_TI:
    LR = 5e-3
    def ensure_placeholder_token(text_tok, text_enc, token: str) -> int:
        """Guarantee `token` is a single id; add as special if needed; resize embeddings."""
        # Already single-id?
        ids = text_tok(token, add_special_tokens=False).input_ids
        if len(ids) == 1:
            return ids[0]

        add_tokens(text_tok, text_enc, [token], device)
        # Re-check
        ids = text_tok(token, add_special_tokens=False).input_ids
        assert len(ids) == 1, f"Still not single token: {token} -> {ids}"
        return ids[0]

    train_selected_tokens(
        names=['blastoise', 'gengar', 'pikachu', 'charizard'],
        steps_per_image=40,   # heuristic; bump to 50–70 for small folders, lower for big
        min_steps=800,        # floor
        max_steps=1600,       # cap
        batch_size=2,         # MPS-safe; raise only if you see headroom
        lr=LR,              # standard for TI
        overwrite_existing=OVERWRITE_EXISTING_TOKENS,
    )


### Generate an example Pokemon image

In [ ]:
from PIL import Image
pose = Image.open("./datasets/pokemon-images/horse_rider.png").convert("RGB").resize((512,512))   # pick one of the rider diagrams above
horse = Image.open("./datasets/pokemon-images/horse.png").convert("RGB").resize((512,512))  # full-body, side/¾ view


In [ ]:
# 1) Clean pipe
import os, torch
from pathlib import Path
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

if torch.backends.mps.is_available():
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK","1")
device = str(get_device())
dtype = torch.float16 if device=="cuda" else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=dtype, safety_checker=None, feature_extractor=None
).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# 2) Load your LoRA file and fuse it (works with A1111 keys)
LORA_DIR = Path("results/pokemon-images/lora_peft_minimal")   # dir containing adapter_model.safetensors
WEIGHT = "adapter_model.safetensors"                          # or pytorch_lora_weights.safetensors
pipe.load_lora_weights(str(LORA_DIR), weight_name=WEIGHT)     # loads weights
pipe.fuse_lora(lora_scale=0.9)                                # bake them in; no adapters needed

# 3) Inject TI vectors you trained
def ensure_single_token(tokenizer, text_encoder, token: str) -> int:
    ids = tokenizer(token, add_special_tokens=False).input_ids
    if len(ids)==1: return ids[0]
    tokenizer.add_special_tokens({"additional_special_tokens":[token]})
    if device=="mps": text_encoder.to("cpu")
    text_encoder.resize_token_embeddings(len(tokenizer))
    if device=="mps": text_encoder.to(device)
    ids = tokenizer(token, add_special_tokens=False).input_ids
    assert len(ids)==1, f"Still splits: {token} -> {ids}"
    return ids[0]

TI_DIR = Path("./results/pokemon-images/ti_embeddings")
if TI_DIR.exists():
    for p in sorted(TI_DIR.glob("*.pt")):
        payload = torch.load(p, map_location="cpu")
        token, vec = payload["token"], payload["vector"].float()
        tid = ensure_single_token(pipe.tokenizer, pipe.text_encoder, token)
        with torch.no_grad():
            w = pipe.text_encoder.get_input_embeddings().weight
            assert vec.numel()==w.shape[1], f"{p.name}: TI dim mismatch"
            w[tid].copy_(vec.to(w.dtype))

# 4) Build a simple prompt (or use your build_prompt/POKEDESC)
pos = "<pk_pikachu>, Pikachu is surfing with waves over his head"
neg = "multiple subjects, low quality, worst quality, blurry, jpeg artifacts, extra heads"

# 5) Render
g = torch.Generator(device=device).manual_seed(1234)

img = pipe(prompt=pos, negative_prompt=neg, num_inference_steps=30, guidance_scale=5,
           width=512, height=512, generator=g).images[0]
Path("./results").mkdir(exist_ok=True)
img.save("./results/pikachu_lora_ti.png")
print("Saved → ./results/pikachu_lora_ti.png")


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
No LoRA keys associated to UNet2DConditionModel found with the prefix='unet'. This is safe to ignore if LoRA state dict didn't originally have any UNet2DConditionModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/is

  0%|          | 0/30 [00:00<?, ?it/s]

Saved → ./results/pikachu_lora_ti.png


In [79]:
pos = "<pk_pikachu> dancing at a redneck bar with 3 taller people dressed like cowboys, overhead view, (full body:1.2)"
neg = "banners, writing, low quality, worst quality, blurry, jpeg artifacts, extra heads"

img = pipe(prompt=pos, negative_prompt=neg, num_inference_steps=50, guidance_scale=10,
           width=512, height=512, generator=g).images[0]
Path("./results").mkdir(exist_ok=True)
img.save("./results/pikachu_redneck.png")
print("Saved → ./results/pikachu_redneck.png")

  0%|          | 0/50 [00:00<?, ?it/s]

Saved → ./results/pikachu_redneck.png


### Define mapping from Pokemon features to MTG features
This helps the translation from Pokemon to MTG creature

In [ ]:

def scaled_stat(value: int, lo: int = 1, hi: int = 12, base: int = 50, spread: int = 100) -> int:
    """Map a Pokémon base stat (~0-255) to an MTG-ish P/T scale.

    Why: Keeps numbers small and printable while preserving relative differences.
    """
    x = (value - base) / spread
    y = (hi - lo) / 2 * x + (lo + hi) / 2
    return max(lo, min(hi, int(round(y))))


def mana_from_stats(power: int, toughness: int, colors: List[str]) -> str:
    cmc = max(1, math.ceil((power + toughness) / 3)) + max(0, len([c for c in colors if c != "C"]) - 1)
    colored = [c for c in colors if c != "C"] or ["C"]
    base_color = colored[0]
    generic = max(0, cmc - len(colored))
    return "".join([f"{{{generic}}}" if generic else "", *[f"{{{base_color}}}" for _ in range(cmc - generic)]]) or "{1}"


def color_identity(types: List[str]) -> List[str]:
    ids = []
    for t in types:
        c = TYPE_TO_COLORS.get(t.lower(), "C")
        if c not in ids:
            ids.extend(c)
    return ids or ["C"]


def mtg_keywords(abilities: List[str]) -> List[str]:
    out: List[str] = []
    for a in abilities:
        k = ABILITY_MAP.get(a)
        if k and k not in out:
            out.append(k)
    return out


def build_card(p: PokemonInfo) -> PokemonCard:
    colors = color_identity(p.types)

    power = scaled_stat(p.base_stats.get("attack", 60))
    toughness = scaled_stat(p.base_stats.get("defense", 60))

    rarity = "Mythic Rare" if (p.is_mythical or p.is_legendary) else "Rare"

    subtype = TYPE_TO_MTG_SUBTYPE.get(p.types[0], "Beast") if p.types else "Beast"
    type_line = ("Legendary " if (p.is_mythical or p.is_legendary) else "") + f"Creature — {subtype}"

    kws = mtg_keywords(p.abilities)
    rules: List[str] = []
    if kws:
        rules.append(", ".join(sorted(set(kws))))
    if len(p.types) > 1:
        rules.append(f"{p.types[1]} Alignment — {p.name} gets +1/+1 until end of turn.")

    pos, neg = compose_mtg_art_prompt(p)
    mana = mana_from_stats(power, toughness, colors)

    return PokemonCard(
        name=p.name,
        mana_cost=mana,
        colors=colors,
        type_line=type_line,
        rarity=rarity,
        rules_text=rules or ["—"],
        power=power,
        toughness=toughness,
        flavor_text=p.flavor_text,
        image_prompt=pos,
        negative_prompt=neg,
        lora_trigger_token=f"<{p.name.lower()}-lora>",
    )

# -----------------------------
# Prompting
# -----------------------------

# /notebooks/mtg_pokemon_ability_map.py




def compose_core_description(p: PokemonInfo) -> str:
    """Concise, vivid description oriented for diffusion prompts."""
    archetype = []
    if "Ghost" in p.types:
        archetype.append("spectral")
    if "Poison" in p.types:
        archetype.append("toxic")
    if "Dragon" in p.types:
        archetype.append("draconic")
    if "Fire" in p.types:
        archetype.append("fiery")
    if "Water" in p.types:
        archetype.append("aquatic")
    if "Electric" in p.types:
        archetype.append("crackling with electricity")

    species_hint = f"{p.name}, a {', '.join(p.types)}-type creature"
    flair = ", ".join(archetype) if archetype else "powerful"

    return (
        f"{species_hint}. A {flair} fantasy monster with iconic silhouette, coherent anatomy, expressive face, "
        f"dynamic lighting, painterly brushwork, detailed textures, high fidelity, game concept art"
    )


def compose_mtg_art_prompt(p: PokemonInfo) -> Tuple[str, str]:
    core = compose_core_description(p)
    setting_bits = [
        "Magic: the Gathering-style fantasy illustration",
        "centered composition",
        "dramatic rim lighting",
        "shallow depth of field",
        "8k, highly detailed, volumetric lighting",
        "textless art, no borders, no logo",
    ]
    positive = "; ".join([core] + setting_bits)

    negative = ", ".join(
        [
            "blurry", "low-res", "deformed", "disfigured", "extra limbs", "out-of-frame",
            "watermark", "signature", "text", "logo", "nsfw",
        ]
    )
    return positive, negative



### Environment & Hardware Notes

* **Developed on macOS + M‑series (MPS) baseline**, designed to work all major families of GPUs or CPU.

* **Versions that behave well**

  * Python **3.10/3.11**
  * PyTorch **2.5.1**, torchvision **0.20.1**
  * diffusers **0.35.1**, transformers **4.44+**, peft **0.11+**, safetensors **0.4+**

* **MPS fallback**

  * Set **before importing torch**:

    ```python
    import os
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # set first
    import torch
    ```
  * Not all ops auto‑fallback; run **tokenizer resize on CPU** and keep the rest on MPS.

* **Data loading**

  * On MPS: `num_workers=0`, `pin_memory=False`, `persistent_workers=False`.
  * Enable `cache_latents=True` to avoid re‑encoding with the VAE each step.

* **Precision & dtypes**

  * Use **float32** on MPS (no fp16 autocast). Keep AMP only for CUDA.

* **Schedulers**

  * Prefer **Euler Ancestral** for MPS inference. DPMSolver can hit unsupported ops more often.

* **Device placement (key gotchas)**

  * **Text encoder resize** after adding tokens must be done on **CPU**:

    ```python
    new_size = tok.vocab_size + len(tok.get_added_vocab())
    text_enc.to("cpu"); text_enc.resize_token_embeddings(new_size); text_enc.to("mps")
    ```
  * During TI: freeze UNet/TE, set `emb.weight.requires_grad_(True)`, and **mask grads** to the specific token row **after** `backward()`.

* **Batch/steps heuristics (M4 36 GB)**

  * Start `batch_size=2`; try 3 if headroom.
  * TI: \~**800–1600 steps/token** (or `~40–60 × images_per_token`), LR `5e-3`.

* **Determinism**

  * Set seeds for both CPU/MPS:

    ```python
    import torch, random
    random.seed(42); torch.manual_seed(42)
    ```

* **Settings for better performance**</br>
`cache_latents=True`, `num_workers=0`, `pin_memory=False`.</br>
Batch size heuristics; step heuristics (`steps_per_image`, min/max caps).</br>
Timer hooks to measure H2D/UNet/Text blocks per step.</br>
